# URL Uploader Bot — Google Colab

A clean Colab launcher for the **`main`** branch of `Telebotfaroff/URL-uploader`.

## Before you run
1. Open **Colab → Secrets** (key icon in the left sidebar).
2. Add `BOT_TOKEN`, `API_ID`, `API_HASH`, and `OWNER_ID` as secrets.
3. Enable notebook access for each secret.
4. Run the cells from top to bottom. Keep this notebook private; never print or paste secret values into cells.

`OWNER_ID` must be your numeric Telegram user ID. Optional settings such as `DATABASE_URL`, `LOG_CHANNEL`, `ADMIN`, `BANNED_USERS`, `HTTP_PROXY`, and `CHUNK_SIZE` can also be added as Colab secrets if you use them.

In [ ]:
# Read credentials from Colab Secrets without displaying them.
from google.colab import userdata
import os

REQUIRED_SECRETS = ("BOT_TOKEN", "API_ID", "API_HASH", "OWNER_ID")
missing = []

for key in REQUIRED_SECRETS:
    try:
        value = userdata.get(key)
    except Exception:
        value = None
    if not value:
        missing.append(key)
    else:
        os.environ[key] = str(value).strip()

if missing:
    raise RuntimeError(
        "Missing Colab Secrets: " + ", ".join(missing) +
        ". Add them in the Colab Secrets panel and enable notebook access."
    )

# Optional configuration: only set these if the corresponding secret exists.
OPTIONAL_SECRETS = (
    "DATABASE_URL", "LOG_CHANNEL", "ADMIN", "BANNED_USERS",
    "HTTP_PROXY", "CHUNK_SIZE", "DEF_THUMB_NAIL_VID_S"
)
for key in OPTIONAL_SECRETS:
    try:
        value = userdata.get(key)
    except Exception:
        value = None
    if value:
        os.environ[key] = str(value).strip()

print("Required credentials loaded securely.")

## 1. Install system and Python dependencies

This installs FFmpeg and the repository's requirements. Pyrogram and TgCrypto are installed explicitly because the bot imports Pyrogram directly.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!python -m pip install -q --upgrade pip
!python -m pip install -q -r https://raw.githubusercontent.com/Telebotfaroff/URL-uploader/main/requirements.txt
!python -m pip install -q "pyrogram==2.0.106" tgcrypto

## 3. Verify and start the bot

The startup cell checks that Telegram accepts your bot token and prints the bot username and configured owner ID before launching. The bot is **owner-only**: send `/start` from the Telegram account whose numeric user ID matches `OWNER_ID`. Keep an eye on the cell output for startup errors. If you see a conflict/another active instance error, stop any other running copy of this bot using the same token.

This cell stays active while the bot is running. Stop the cell or disconnect the runtime to stop it. Colab is temporary hosting and may disconnect.

In [ ]:
from pathlib import Path
import shutil

REPO_DIR = Path("/content/URL-uploader")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

!git clone --depth 1 --branch main https://github.com/Telebotfaroff/URL-uploader.git /content/URL-uploader
%cd /content/URL-uploader

print("Repository ready on branch main.")

## 3. Verify, start, and view live logs

The startup cell checks the token and starts the bot with **live stdout/stderr logs displayed directly in Colab**. The same output is appended to `colab_bot.log` so you can inspect it later with `!tail -n 200 /content/URL-uploader/colab_bot.log`.

This bot is **owner-only**: send `/start` from the Telegram account whose numeric user ID matches `OWNER_ID`. If you see a conflict/another active instance error, stop any other running copy using the same token. Stop the cell to stop the bot. Colab runtimes are temporary.

In [ ]:
import os
import subprocess
import sys
import requests
from datetime import datetime

# Preflight: confirm the token works and show which bot will be started.
token = os.environ["BOT_TOKEN"]
owner_id = int(os.environ["OWNER_ID"])
if owner_id <= 0:
    raise ValueError("OWNER_ID must be your numeric Telegram user ID.")

response = requests.get(
    f"https://api.telegram.org/bot{token}/getMe",
    timeout=20,
)
response.raise_for_status()
result = response.json()
if not result.get("ok"):
    raise RuntimeError("Telegram rejected BOT_TOKEN. Check the BOT_TOKEN Colab Secret.")
bot_info = result["result"]
print(f"Token verified. Bot: @{bot_info.get('username', 'unknown')}", flush=True)
print(f"Allowed owner Telegram user ID: {owner_id}", flush=True)
print("This bot is owner-only; /start must be sent by the Telegram account matching OWNER_ID.", flush=True)

os.makedirs("DOWNLOADS", exist_ok=True)
LOG_PATH = "/content/URL-uploader/colab_bot.log"
print(f"\nStarting bot. Live logs will appear below and be saved to: {LOG_PATH}\n" + "-" * 72, flush=True)

process = subprocess.Popen(
    [sys.executable, "-u", "bot.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=os.environ.copy(),
)

try:
    with open(LOG_PATH, "a", encoding="utf-8", buffering=1) as log_file:
        log_file.write(f"\n{'=' * 24} Session started {datetime.now().isoformat(timespec='seconds')} {'=' * 24}\n")
        for line in process.stdout:
            log_file.write(line)
            print(line, end="", flush=True)
    return_code = process.wait()
    print(f"\nBot process exited with code {return_code}.", flush=True)
except KeyboardInterrupt:
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()
    print("\nBot stopped.", flush=True)
finally:
    if process.stdout:
        process.stdout.close()

print("To inspect the saved log later in this runtime, run: !tail -n 200 /content/URL-uploader/colab_bot.log")